# SAR-to-EO Image Translation — GalaxEye Assignment
**Model:** Pix2Pix U-Net + PatchGAN  
**Loss:** L1 + Adversarial + FFT Frequency + VGG Perceptual  
**Dataset:** Sentinel-1&2 terrain-split (Kaggle)


In [ ]:
# ============================================================
# CELL 1: Check GPU and install dependencies
# ============================================================
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout)

import torch
print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU            : {torch.cuda.get_device_name(0)}')
    print(f'VRAM           : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# ============================================================
# CELL 2: Install required packages
# ============================================================
!pip install lpips pytorch-fid scikit-image -q

In [ ]:
# ============================================================
# CELL 3: Upload and extract project code
# Instructions: Upload sar2eo.zip via Kaggle 'Add Data' or
# paste the code cells directly from your GitHub repo.
# ============================================================
import os

# Option A: Clone from GitHub (recommended)
# !git clone https://github.com/<your-username>/sar2eo.git
# os.chdir('sar2eo')

# Option B: If uploaded as zip
# !unzip /kaggle/input/sar2eo-code/sar2eo.zip -d /kaggle/working/
# os.chdir('/kaggle/working/sar2eo')

print('Working directory:', os.getcwd())
print('Files:', os.listdir('.'))

In [ ]:
# ============================================================
# CELL 4: Check dataset structure
# ============================================================
import os

DATASET_PATH = '/kaggle/input/sentinel12-image-pairs-segregated-by-terrain'

print('Dataset contents:')
for item in sorted(os.listdir(DATASET_PATH)):
    item_path = os.path.join(DATASET_PATH, item)
    if os.path.isdir(item_path):
        contents = os.listdir(item_path)
        print(f'  {item}/ ({len(contents)} subdirs/files)')
        for sub in sorted(contents):
            sub_path = os.path.join(item_path, sub)
            if os.path.isdir(sub_path):
                n = len(os.listdir(sub_path))
                print(f'    {sub}/ ({n} files)')

In [ ]:
# ============================================================
# CELL 5: Visualise a few SAR/EO pairs before training
# (Data analysis — required for the report)
# ============================================================
import os
import random
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

DATASET_PATH = '/kaggle/input/sentinel12-image-pairs-segregated-by-terrain'
TERRAINS = ['barren', 'grassland', 'agricultural', 'urban']

fig, axes = plt.subplots(len(TERRAINS), 4, figsize=(16, 4 * len(TERRAINS)))
col_labels = ['SAR (VV)', 'EO (RGB)', 'SAR histogram', 'EO histogram']

for row_idx, terrain in enumerate(TERRAINS):
    # Find s1/s2 dirs (handle different naming)
    terrain_dir = os.path.join(DATASET_PATH, terrain)
    dirs = os.listdir(terrain_dir) if os.path.exists(terrain_dir) else []
    
    s1_dir = os.path.join(terrain_dir, 's1')
    s2_dir = os.path.join(terrain_dir, 's2')
    
    if not os.path.exists(s1_dir):
        print(f'Terrain {terrain}: dirs = {dirs}')
        continue
    
    sar_files = sorted(os.listdir(s1_dir))
    sample = random.choice(sar_files)
    
    sar_img = np.array(Image.open(os.path.join(s1_dir, sample)).convert('L'))
    eo_img  = np.array(Image.open(os.path.join(s2_dir, sample)).convert('RGB'))
    
    axes[row_idx][0].imshow(sar_img, cmap='gray')
    axes[row_idx][0].set_title(f'{terrain} — SAR', fontsize=10)
    axes[row_idx][0].axis('off')
    
    axes[row_idx][1].imshow(eo_img)
    axes[row_idx][1].set_title(f'{terrain} — EO', fontsize=10)
    axes[row_idx][1].axis('off')
    
    axes[row_idx][2].hist(sar_img.flatten(), bins=50, color='gray', alpha=0.7)
    axes[row_idx][2].set_title('SAR pixel distribution', fontsize=9)
    axes[row_idx][2].set_xlabel('Pixel value [0-255]')
    
    for ch, col in enumerate(['r', 'g', 'b']):
        axes[row_idx][3].hist(eo_img[:,:,ch].flatten(), bins=50,
                              color=col, alpha=0.5, label=col.upper())
    axes[row_idx][3].set_title('EO pixel distribution', fontsize=9)
    axes[row_idx][3].set_xlabel('Pixel value [0-255]')
    axes[row_idx][3].legend(fontsize=8)

plt.suptitle('SAR vs EO Data Analysis by Terrain Class', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/data_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('Data analysis saved.')

In [ ]:
# ============================================================
# CELL 6: Update config for Kaggle environment
# ============================================================
import yaml

with open('config.yaml') as f:
    cfg = yaml.safe_load(f)

# Point to Kaggle dataset
cfg['data']['dataset_type'] = 'kaggle'
cfg['data']['kaggle_root']  = '/kaggle/input/sentinel12-image-pairs-segregated-by-terrain'
cfg['data']['subset_size']  = None   # Use all available data on Kaggle

# Kaggle T4 has 16GB — can use batch_size=4 comfortably
cfg['training']['batch_size'] = 4
cfg['training']['epochs']     = 100

with open('config.yaml', 'w') as f:
    yaml.dump(cfg, f, default_flow_style=False)

print('Config updated for Kaggle:')
print(f"  dataset_type : {cfg['data']['dataset_type']}")
print(f"  kaggle_root  : {cfg['data']['kaggle_root']}")
print(f"  batch_size   : {cfg['training']['batch_size']}")
print(f"  epochs       : {cfg['training']['epochs']}")

In [ ]:
# ============================================================
# CELL 7: Smoke test — 2 epochs on a tiny subset
# Run this BEFORE the full training to catch any errors.
# ============================================================
import yaml

# Temporarily use tiny subset for smoke test
with open('config.yaml') as f:
    cfg_orig = yaml.safe_load(f)

cfg_smoke = cfg_orig.copy()
cfg_smoke['data']     = dict(cfg_orig['data'])
cfg_smoke['training'] = dict(cfg_orig['training'])
cfg_smoke['data']['subset_size']  = 100
cfg_smoke['training']['epochs']   = 2
cfg_smoke['training']['val_freq'] = 1
cfg_smoke['training']['save_freq'] = 2
cfg_smoke['active_ablation']      = 'full'

with open('config_smoke.yaml', 'w') as f:
    yaml.dump(cfg_smoke, f)

print('Running smoke test (2 epochs, 100 pairs)...')
!python train.py --config config_smoke.yaml --ablation full
print('Smoke test complete!')

In [ ]:
# ============================================================
# CELL 8a: ABLATION CONFIG A — L1 only (baseline)
# ============================================================
print('Training Config A: L1 only...')
!python train.py --config config.yaml --ablation l1_only
print('Config A done!')

In [ ]:
# ============================================================
# CELL 8b: ABLATION CONFIG B — L1 + Adversarial
# ============================================================
print('Training Config B: L1 + Adversarial...')
!python train.py --config config.yaml --ablation l1_adv
print('Config B done!')

In [ ]:
# ============================================================
# CELL 8c: ABLATION CONFIG C — L1 + Adversarial + FFT
# ============================================================
print('Training Config C: L1 + Adversarial + FFT...')
!python train.py --config config.yaml --ablation l1_adv_fft
print('Config C done!')

In [ ]:
# ============================================================
# CELL 8d: MAIN MODEL — Config D: Full loss stack
# L1 + Adversarial + FFT + VGG Perceptual
# ============================================================
print('Training Config D (MAIN MODEL): Full loss stack...')
!python train.py --config config.yaml --ablation full
print('Config D done!')

In [ ]:
# ============================================================
# CELL 9: Evaluate all configs on test split
# ============================================================
for ablation in ['l1_only', 'l1_adv', 'l1_adv_fft', 'full']:
    ckpt = f'checkpoints/{ablation}/best.pth'
    if not os.path.exists(ckpt):
        print(f'Skipping {ablation} — no checkpoint found')
        continue
    print(f'\n=== Evaluating: {ablation} ===')
    !python eval.py --config config.yaml --weights {ckpt} --split test

print('\nAll evaluations complete!')

In [ ]:
# ============================================================
# CELL 10: Save outputs for download
# ============================================================
import shutil, os

# Zip checkpoints + outputs for download
shutil.make_archive('/kaggle/working/sar2eo_results', 'zip', '.', 'outputs')
shutil.make_archive('/kaggle/working/sar2eo_checkpoints', 'zip', '.', 'checkpoints')

print('Results zipped:')
print('  /kaggle/working/sar2eo_results.zip')
print('  /kaggle/working/sar2eo_checkpoints.zip')
print('Download these from the Kaggle notebook output panel.')